In [3]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install scikeras

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
!pip install kagglehub[pandas-datasets]

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# =========================
# INSTALL (run once if needed)
# =========================
# pip install kagglehub[pandas-datasets] scikeras tensorflow scikit-learn pandas matplotlib

# =========================
# IMPORTS
# =========================
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from scikeras.wrappers import KerasClassifier

# =========================
# LOAD DATASET
# =========================
file_path = "Churn_Modelling.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shrutimechlearn/churn-modelling",
    file_path
)

print("Dataset Loaded")
print(df.head())

# =========================
# PREPROCESSING
# =========================
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Encode Gender
le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])

# One-hot encode Geography
df = pd.get_dummies(df, columns=["Geography"], drop_first=True)

# Features & target
X = df.drop("Exited", axis=1)
y = df["Exited"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# BUILD MODEL FUNCTION
# =========================
def build_model(neurons=32, dropout_rate=0.3, optimizer='adam'):
    model = keras.Sequential()

    model.add(layers.Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(neurons, activation='relu'))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# =========================
# SCIKERAS WRAPPER
# =========================
model = KerasClassifier(
    model=build_model,
    verbose=0
)

# =========================
# GRID SEARCH PARAMETERS
# =========================
param_grid = {
    'model__neurons': [16, 32],
    'model__dropout_rate': [0.2, 0.3],
    'optimizer': ['adam', 'rmsprop'],
    'batch_size': [16, 32],
    'epochs': [50]
}

# =========================
# CROSS VALIDATION
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1
)

print("Running Grid Search...")
grid_result = grid.fit(X_train, y_train)

print("\nBest Parameters:", grid_result.best_params_)
print("Best Cross-Validation Accuracy:", grid_result.best_score_)

# =========================
# MODEL CHECKPOINT
# =========================
checkpoint = keras.callbacks.ModelCheckpoint(
    "best_churn_model.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# =========================
# TRAIN FINAL MODEL
# =========================
best_params = grid_result.best_params_

final_model = build_model(
    neurons=best_params['model__neurons'],
    dropout_rate=best_params['model__dropout_rate'],
    optimizer=best_params['optimizer']
)

history = final_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    callbacks=[checkpoint],
    verbose=1
)

# =========================
# EVALUATION
# =========================
loss, accuracy = final_model.evaluate(X_test, y_test)

print("\nFinal Test Accuracy:", accuracy)

# =========================
# OPTIONAL: LOAD BEST MODEL
# =========================
best_model = keras.models.load_model("best_churn_model.h5")
best_loss, best_acc = best_model.evaluate(X_test, y_test)

print("Best Saved Model Accuracy:", best_acc)

/opt/intel/oneapi/intelpython/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-06 20:22:29.703768: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 20:22:29.734237: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/tmp/ipykernel_867544/2707093574.py:28: DeprecationWarning:

100%|████████████████████████████████████████| 669k/669k [00:01<00:00, 361kB/s]

Dataset Loaded
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0 


2026-05-06 20:22:36.208099: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 20:22:36.244889: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-06 20:22:36.670103: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 20:22:36.672870: 


Best Parameters: {'batch_size': 32, 'epochs': 50, 'model__dropout_rate': 0.2, 'model__neurons': 32, 'optimizer': 'rmsprop'}
Best Cross-Validation Accuracy: 0.8619999999999999
Epoch 1/50


...

200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8517 - loss: 0.3705 - val_accuracy: 0.8581 - val_loss: 0.3522
Epoch 12/50

Epoch 12: val_accuracy improved from 0.85812 to 0.85875, saving model to best_churn_model.h5
- loss: 0.3617    

200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8519 - loss: 0.3687 - val_accuracy: 0.8587 - val_loss: 0.3502
Epoch 13/50

Epoch 13: val_accuracy did not improve from 0.85875
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8498 - loss: 0.3647 - val_accuracy: 0.8550 - val_loss: 0.3489
Epoch 14/50

Epoch 14: val_accuracy did not improve from 0.85875
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8508 - loss: 0.3686 - val_accuracy: 0.8569 - val_loss: 0.3490
Epoch 15/50

Epoch 15: val_accuracy did not improve from 0.85875
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8575 - loss: 0.3599 - val_accuracy: 0.8531 - val_loss: 0.3487
Epoch 16/50

Epoch 16: val_accuracy did not improve from 0.85875
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8561 - loss: 0.3538 - val_accuracy: 0.8562 - val_loss: 0.3468
Epoch 17/50

Epoch 17: val_accuracy did not improve from 0.85875
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8556 - loss: 0.3559 - val_accuracy:

200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8658 - loss: 0.3451 - val_accuracy: 0.8600 - val_loss: 0.3409
Epoch 50/50

Epoch 50: val_accuracy improved from 0.86000 to 0.86063, saving model to best_churn_model.h5
- loss: 0.3179   

200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8677 - loss: 0.3346 - val_accuracy: 0.8606 - val_loss: 0.3408
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8595 - loss: 0.3372  



Final Test Accuracy: 0.859499990940094
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8595 - loss: 0.3372  
Best Saved Model Accuracy: 0.859499990940094
